In [ ]:
# Installation des dépendances nécessaires (exécuter une seule fois)
%pip install -q trl==0.25.1 peft bitsandbytes accelerate transformers datasets sentencepiece huggingface_hub evaluate

# ⚠️ IMPORTANT pour Llama-3: Vous devez avoir un token Hugging Face valide
# 1. Créez un compte sur https://huggingface.co
# 2. Acceptez les conditions d'utilisation de Meta-Llama-3-8B-Instruct
# 3. Créez un token sur https://huggingface.co/settings/tokens
# 4. Utilisez ce token ci-dessous

from huggingface_hub import login
try:
    login()  # ⚠️ OBLIGATOIRE pour Llama-3: entrez votre token Hugging Face ici
    print("✅ Authentification Hugging Face réussie")
except Exception as e:
    print(f"❌ Erreur d'authentification: {e}")
    print("💡 Définissez HF_TOKEN dans l'environnement ou utilisez login() avec votre token")
    print("💡 Llama-3 nécessite un accès gated - acceptez les conditions d'utilisation d'abord")


In [ ]:
# ---------------- Notebook-ready SFT (complete) avec métriques d'évaluation ----------------
# Installez d'abord (une seule fois) si nécessaire :
# !pip install -q trl==0.25.1 peft bitsandbytes accelerate transformers datasets sentencepiece huggingface_hub evaluate

import os
import json
import re
import torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from huggingface_hub import login
from typing import Dict, List, Tuple
import evaluate
from collections import Counter

# --- Small fixes to reduce OOM and fragmentation
# Set this early to reduce fragmentation as suggested by PyTorch errors
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

# ---------------- Hugging Face login (interactive)
print("Login to Hugging Face (paste token when prompted)...")
try:
    login()  # colle ton token HuggingFace (scope read)
except:
    print("Login skipped (token may be set via environment)")

# ---------------- Config générale (modèle)
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

# Template aligné avec Llama-3 chat format
def format_chat_template(system_prompt: str, instruction: str, response: str, tokenizer=None) -> str:
    """Formate le prompt selon le template de Llama-3"""
    # Si le tokenizer a un chat_template, l'utiliser (plus fiable)
    if tokenizer and hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response}
        ]
        try:
            formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            return formatted
        except:
            pass  # Fallback au format manuel
    
    # Format manuel Llama-3 (fallback)
    formatted = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
    return formatted

def formatting_prompts_func(example, tokenizer=None):
    """Convertit un batch JSONL → texte formatté"""
    texts = []
    if isinstance(example.get("instruction"), list):
        length = len(example["instruction"])
        for i in range(length):
            formatted = format_chat_template(
                example["system_prompt"][i],
                example["instruction"][i],
                example["response"][i],
                tokenizer=tokenizer
            )
            texts.append(formatted)
    else:
        formatted = format_chat_template(
            example.get("system_prompt", ""),
            example.get("instruction", ""),
            example.get("response", ""),
            tokenizer=tokenizer
        )
        texts.append(formatted)
    return {"text": texts}

def make_safe_data_collator(tokenizer):
    base_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    def collator(features):
        if isinstance(features, (list, tuple)) and len(features) > 0 and isinstance(features[0], dict):
            for f in features:
                f.pop("text", None)
        return base_collator(features)
    return collator

# ---------------- Métriques d'évaluation personnalisées
def extract_json_from_text(text: str) -> dict:
    """Extrait le JSON d'un texte, même s'il y a du texte autour"""
    # Stratégie 1: Chercher un bloc JSON complet
    match = re.search(r"\{.*\}", text.strip(), re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    # Stratégie 2: Chercher des paires clé-valeur
    found_keys = {}
    patterns = [
        r'"(\w+)":\s*"([^"]*)"',
        r'"(\w+)":\s*(\d+)',
        r'"(\w+)":\s*(true|false)',
    ]
    for pattern in patterns:
        matches = re.finditer(pattern, text, re.IGNORECASE)
        for match in matches:
            key = match.group(1)
            if len(match.groups()) > 1:
                value = match.group(2)
                if value.lower() in ['true', 'false']:
                    found_keys[key] = value.lower() == 'true'
                elif value.isdigit():
                    found_keys[key] = int(value)
                else:
                    found_keys[key] = value
    return found_keys

def compute_json_accuracy(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Calcule l'accuracy pour la génération de JSON"""
    json_valid = 0
    json_parseable = 0
    key_match = 0
    exact_match = 0
    total = len(predictions)
    
    expected_keys_by_agent = {
        "orchestrator": {"AGENT_CIBLE", "COMMANDE"},
        "researcher": set(),  # Pas de JSON attendu
        "code_writer": set(),  # Code Python
        "critic": set(),  # Texte libre
    }
    
    for pred, ref in zip(predictions, references):
        try:
            # Essayer de parser la référence comme JSON
            ref_json = json.loads(ref) if isinstance(ref, str) else ref
            ref_is_json = isinstance(ref_json, dict)
        except:
            ref_json = None
            ref_is_json = False
        
        # Extraire le JSON de la prédiction
        pred_json = extract_json_from_text(pred)
        
        if pred_json:
            json_parseable += 1
            try:
                json.loads(json.dumps(pred_json))  # Validation JSON
                json_valid += 1
            except:
                pass
            
            if ref_is_json and ref_json:
                # Vérifier les clés attendues
                if set(pred_json.keys()) == set(ref_json.keys()):
                    key_match += 1
                # Exact match
                if pred_json == ref_json:
                    exact_match += 1
    
    return {
        "json_valid_rate": json_valid / total if total > 0 else 0.0,
        "json_parseable_rate": json_parseable / total if total > 0 else 0.0,
        "key_match_rate": key_match / total if total > 0 else 0.0,
        "exact_match_rate": exact_match / total if total > 0 else 0.0,
    }

def compute_token_accuracy(predictions, labels) -> Dict[str, float]:
    """Calcule l'accuracy au niveau des tokens"""
    # Convertir en numpy arrays si nécessaire
    if isinstance(predictions, list):
        predictions = np.array(predictions)
    if isinstance(labels, list):
        labels = np.array(labels)
    
    # Si predictions sont des logits, prendre argmax
    if len(predictions.shape) == 3:  # (batch, seq_len, vocab_size)
        predictions = np.argmax(predictions, axis=-1)
    
    # S'assurer que les shapes correspondent
    if predictions.shape != labels.shape:
        min_len = min(predictions.shape[1], labels.shape[1])
        predictions = predictions[:, :min_len]
        labels = labels[:, :min_len]
    
    total_tokens = 0
    correct_tokens = 0
    exact_matches = 0
    
    for pred_seq, label_seq in zip(predictions, labels):
        # Ignorer les tokens de padding (-100)
        valid_mask = label_seq != -100
        if not np.any(valid_mask):
            continue
        
        # Compter les tokens corrects
        valid_preds = pred_seq[valid_mask]
        valid_labels = label_seq[valid_mask]
        
        correct = (valid_preds == valid_labels).sum()
        total_tokens += len(valid_labels)
        correct_tokens += correct
        
        # Vérifier l'exact match (seulement sur les tokens valides)
        if np.array_equal(valid_preds, valid_labels):
            exact_matches += 1
    
    accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_rate = exact_matches / len(predictions) if len(predictions) > 0 else 0.0
    
    return {
        "token_accuracy": float(accuracy),
        "exact_match_rate": float(exact_match_rate),
    }

def compute_f1_score(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Calcule le F1 score basé sur le chevauchement des mots"""
    def get_tokens(text: str) -> List[str]:
        """Tokenise le texte en mots (simple split)"""
        return text.lower().split()
    
    def compute_precision_recall_f1(pred_tokens: List[str], ref_tokens: List[str]) -> Tuple[float, float, float]:
        """Calcule precision, recall et F1"""
        if len(pred_tokens) == 0 and len(ref_tokens) == 0:
            return 1.0, 1.0, 1.0
        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            return 0.0, 0.0, 0.0
        
        pred_counter = Counter(pred_tokens)
        ref_counter = Counter(ref_tokens)
        
        # Intersection des tokens
        common_tokens = set(pred_tokens) & set(ref_tokens)
        if len(common_tokens) == 0:
            return 0.0, 0.0, 0.0
        
        # Compter les occurrences communes (min pour chaque token)
        true_positives = sum(min(pred_counter[token], ref_counter[token]) for token in common_tokens)
        
        precision = true_positives / len(pred_tokens) if len(pred_tokens) > 0 else 0.0
        recall = true_positives / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
        
        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * (precision * recall) / (precision + recall)
        
        return precision, recall, f1
    
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0
    count = 0
    
    for pred, ref in zip(predictions, references):
        pred_tokens = get_tokens(pred)
        ref_tokens = get_tokens(ref)
        precision, recall, f1 = compute_precision_recall_f1(pred_tokens, ref_tokens)
        total_precision += precision
        total_recall += recall
        total_f1 += f1
        count += 1
    
    avg_precision = total_precision / count if count > 0 else 0.0
    avg_recall = total_recall / count if count > 0 else 0.0
    avg_f1 = total_f1 / count if count > 0 else 0.0
    
    return {
        "precision": avg_precision,
        "recall": avg_recall,
        "f1_score": avg_f1,
    }

def compute_eval_metrics(eval_pred, tokenizer, agent_name: str) -> Dict[str, float]:
    """Calcule les métriques d'évaluation incluant accuracy et F1 score"""
    try:
        predictions, labels = eval_pred
        
        # Limiter le nombre d'exemples pour économiser la mémoire
        max_samples = 500
        if len(predictions) > max_samples:
            # Échantillonner aléatoirement
            indices = np.random.choice(len(predictions), max_samples, replace=False)
            predictions = predictions[indices]
            labels = labels[indices]
        
        # Convertir en numpy si nécessaire
        if not isinstance(predictions, np.ndarray):
            predictions = np.array(predictions)
        if not isinstance(labels, np.ndarray):
            labels = np.array(labels)
        
        # Si predictions sont des logits, prendre argmax pour obtenir les token IDs
        if len(predictions.shape) == 3:  # (batch, seq_len, vocab_size)
            pred_token_ids = np.argmax(predictions, axis=-1)
        else:
            pred_token_ids = predictions
        
        # Calculer l'accuracy au niveau des tokens (utilise les token IDs)
        token_metrics = compute_token_accuracy(pred_token_ids, labels)
        
        # Filtrer les tokens invalides (-100) avant le décodage pour éviter les erreurs
        # Décoder les prédictions et labels pour les métriques textuelles
        decoded_preds = []
        decoded_labels = []
        
        # Décoder batch par batch pour économiser la mémoire
        batch_size = 32  # Décoder par petits batches
        for i in range(0, len(pred_token_ids), batch_size):
            batch_preds = pred_token_ids[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            
            # Filtrer les tokens invalides pour les labels avant décodage
            clean_batch_labels = []
            for label_seq in batch_labels:
                # Remplacer -100 par pad_token_id ou eos_token_id pour le décodage
                clean_seq = label_seq.copy()
                clean_seq[label_seq == -100] = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
                clean_batch_labels.append(clean_seq)
            clean_batch_labels = np.array(clean_batch_labels)
            
            try:
                batch_decoded_preds = tokenizer.batch_decode(batch_preds, skip_special_tokens=True)
                batch_decoded_labels = tokenizer.batch_decode(clean_batch_labels, skip_special_tokens=True)
                decoded_preds.extend(batch_decoded_preds)
                decoded_labels.extend(batch_decoded_labels)
            except Exception as e:
                # Si le décodage échoue, utiliser des chaînes vides
                print(f"Warning: Erreur de décodage pour le batch {i}: {e}")
                decoded_preds.extend([""] * len(batch_preds))
                decoded_labels.extend([""] * len(batch_labels))
        
        # Extraire seulement la partie réponse (après <|assistant|>)
        pred_responses = []
        label_responses = []
        
        for pred, label in zip(decoded_preds, decoded_labels):
            # Extraire la réponse générée
            if "<|assistant|>" in pred:
                pred_resp = pred.split("<|assistant|>")[-1].strip()
            else:
                pred_resp = pred
            pred_responses.append(pred_resp)
            
            if "<|assistant|>" in label:
                label_resp = label.split("<|assistant|>")[-1].strip()
            else:
                label_resp = label
            label_responses.append(label_resp)
        
        # Calculer F1 score, precision, recall
        f1_metrics = compute_f1_score(pred_responses, label_responses)
        
        # Métriques JSON pour orchestrator
        metrics = {}
        if agent_name == "orchestrator":
            json_metrics = compute_json_accuracy(pred_responses, label_responses)
            metrics.update(json_metrics)
        
        # Ajouter toutes les métriques
        metrics.update(token_metrics)
        metrics.update(f1_metrics)
        
        return metrics
    
    except Exception as e:
        # En cas d'erreur, retourner des métriques par défaut
        print(f"⚠️ Erreur dans compute_eval_metrics: {e}")
        return {
            "token_accuracy": 0.0,
            "exact_match_rate": 0.0,
            "f1_score": 0.0,
            "precision": 0.0,
            "recall": 0.0,
        }

# ---------------- Callback pour monitoring
class EvaluationCallback(TrainerCallback):
    """Callback pour évaluer périodiquement le modèle"""
    def __init__(self, eval_dataset, tokenizer, agent_name, eval_steps=500):
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer
        self.agent_name = agent_name
        self.eval_steps = eval_steps
        self.step_count = 0
    
    def on_step_end(self, args, state, control, **kwargs):
        self.step_count += 1
        if self.step_count % self.eval_steps == 0:
            print(f"\n[Step {self.step_count}] Evaluation périodique...")

def run_sft_training(dataset_path, output_dir, agent_name, eval_split=0.1):
    print("\n" + "="*70)
    print(f"🚀 START SFT → agent: {agent_name}")
    print(f"📁 Dataset: {dataset_path}")
    print("="*70 + "\n")

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"[ERREUR] Dataset introuvable : {os.path.abspath(dataset_path)}")

    # Detect GPU availability
    has_cuda = torch.cuda.is_available()
    print(f"💻 GPU disponible : {has_cuda}")
    if has_cuda:
        torch_dtype = torch.float16
        device_map = "auto"
    else:
        torch_dtype = torch.float32
        device_map = {"": "cpu"}

    # ---------------- Model load avec quantization pour Llama-3 (8B est plus grand)
    # Utiliser 4-bit quantization pour économiser la mémoire
    
    if has_cuda:
        # Configuration 4-bit optimisée pour éviter OOM
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch_dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        # OPTIMISATION: Limiter la mémoire GPU pour éviter OOM
        # Sur Kaggle T4 (16GB), on limite à 14GB pour laisser de la marge
        max_memory = {0: "14GiB"}  # Limiter à 14GB sur GPU 0
    else:
        quantization_config = None
        max_memory = None
    
    try:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map=device_map,
            trust_remote_code=True,
            torch_dtype=torch_dtype,
            quantization_config=quantization_config if has_cuda else None,
            low_cpu_mem_usage=True,
            max_memory=max_memory
        )
        print(f"✅ Modèle chargé : {BASE_MODEL_ID}")
        if quantization_config:
            print(f"✅ Quantization 4-bit activée pour économiser la mémoire")
    except Exception as e_load:
        error_msg = str(e_load)
        if "out of memory" in error_msg.lower() or "oom" in error_msg:
            print(f"❌ Erreur de mémoire GPU lors du chargement")
            print(f"💡 Solutions possibles:")
            print(f"   1. Réduire model_max_len à 512 dans le code")
            print(f"   2. Utiliser un GPU avec plus de mémoire")
            print(f"   3. Réduire la taille du dataset")
            raise
        elif "CPU or the disk" in error_msg:
            print(f"⚠️ Le modèle ne tient pas entièrement en GPU")
            print(f"💡 Tentative avec offloading CPU activé...")
            try:
                max_memory = {0: "12GiB", "cpu": "30GiB"}  # Réduire encore plus GPU
                model = AutoModelForCausalLM.from_pretrained(
                    BASE_MODEL_ID,
                    device_map="auto",
                    trust_remote_code=True,
                    torch_dtype=torch_dtype,
                    quantization_config=quantization_config,
                    low_cpu_mem_usage=True,
                    max_memory=max_memory
                )
                print(f"✅ Modèle chargé avec offloading CPU/GPU")
            except Exception as e2:
                print(f"❌ Échec même avec offloading: {e2}")
                raise
        else:
            print(f"⚠️ Échec du chargement : {e_load}")
            print(f"💡 Note: Llama-3 nécessite un accès Hugging Face avec token valide")
            print(f"💡 Vérifiez que vous avez: login() avec un token valide")
            raise

    # Préparer le modèle
    # IMPORTANT: Réactiver gradient checkpointing pour économiser la mémoire avec Llama-3 8B
    # Le modèle quantifié nécessite plus de mémoire qu'attendu
    use_gradient_checkpointing = True  # Réactivé pour éviter OOM
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=use_gradient_checkpointing)
    model.config.use_cache = False
    
    # ⚠️ IMPORTANT: torch.compile() n'est PAS compatible avec les modèles quantifiés + PEFT
    # Ne pas compiler le modèle si quantization est activée
    # La compilation sera désactivée automatiquement
    print("ℹ️ Compilation torch.compile désactivée (incompatible avec quantization + PEFT)")

    # ---------------- Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # OPTIMISATION: Réduire encore plus la longueur maximale pour éviter OOM
    # Llama-3 supporte jusqu'à 8192 tokens, mais on limite à 1024 pour la mémoire
    model_max_len = getattr(model.config, 'max_position_embeddings', None) or getattr(model.config, 'n_positions', None) or getattr(tokenizer, 'model_max_length', 1024)
    # Limiter à 1024 pour éviter OOM (peut être augmenté si mémoire disponible)
    model_max_len = min(int(model_max_len), 1024)
    tokenizer.model_max_length = model_max_len
    print(f"ℹ️ Using model_max_len={model_max_len} for truncation (réduit pour mémoire, Llama-3 supporte jusqu'à 8192)")

    # NOTE: do NOT assign tokenizer onto the model to avoid extra references
    # (was: model.tokenizer = tokenizer)  <-- removed

    # ---------------- Dataset avec split train/val
    # OPTIMISATION: Charger avec streaming si dataset très grand
    dataset = load_dataset("json", data_files={"train": dataset_path}, streaming=False)["train"]
    
    # OPTIMISATION: Limiter la taille du dataset pour tests rapides (optionnel)
    # Décommentez les lignes suivantes pour limiter à 1000 échantillons (pour tests)
    # MAX_SAMPLES = 1000
    # if len(dataset) > MAX_SAMPLES:
    #     print(f"⚠️ Dataset limité à {MAX_SAMPLES} échantillons pour tests rapides")
    #     dataset = dataset.select(range(MAX_SAMPLES))
    
    # Split train/validation
    if len(dataset) > 100:  # Seulement si assez de données
        dataset = dataset.train_test_split(test_size=eval_split, seed=42)
        train_dataset = dataset["train"]
        eval_dataset = dataset["test"]
        
        # OPTIMISATION: Limiter encore plus le dataset d'évaluation
        max_eval_samples = 100  # Réduit de 500 à 100 pour vitesse
        if len(eval_dataset) > max_eval_samples:
            eval_dataset = eval_dataset.select(range(max_eval_samples))
            print(f"⚠️ Dataset d'évaluation limité à {max_eval_samples} échantillons pour vitesse")
        
        print(f"📊 Split: {len(train_dataset)} train, {len(eval_dataset)} validation")
    else:
        train_dataset = dataset
        eval_dataset = None
        print(f"📊 Dataset complet utilisé pour training: {len(train_dataset)} exemples")
    
    # Formatage: premier mapping → texte (avec tokenizer pour utiliser le chat_template)
    # OPTIMISATION: Utiliser batch_size plus grand pour le mapping
    def format_with_tokenizer(example):
        return formatting_prompts_func(example, tokenizer=tokenizer)
    
    # OPTIMISATION: Batch size plus grand et désactiver le cache
    train_dataset = train_dataset.map(
        format_with_tokenizer, 
        batched=True, 
        batch_size=1000,  # Batch size plus grand pour vitesse
        remove_columns=[c for c in train_dataset.column_names if c != "text"],
        load_from_cache_file=False  # Désactiver le cache pour éviter les I/O
    )
    if eval_dataset:
        eval_dataset = eval_dataset.map(
            format_with_tokenizer, 
            batched=True, 
            batch_size=1000,
            remove_columns=[c for c in eval_dataset.column_names if c != "text"],
            load_from_cache_file=False
        )
    
    print(f"📝 Exemple formaté (train) : {train_dataset[0]['text'][:200]}...")

    # --- Tokenize + truncate to model_max_len to avoid sequences longer than model supports
    # OPTIMISATION: Tokenization plus rapide
    def tokenize_and_prepare_labels(batch):
        tokenized = tokenizer(
            batch["text"], 
            truncation=True, 
            max_length=model_max_len, 
            return_attention_mask=True,
            padding=False  # Pas de padding ici, fait par le collator
        )
        # For causal LM labels == input_ids
        tokenized["labels"] = tokenized["input_ids"].copy()  # Plus rapide que list comprehension
        return tokenized

    # OPTIMISATION: Batch size plus grand et désactiver le cache
    train_dataset = train_dataset.map(
        tokenize_and_prepare_labels, 
        batched=True, 
        batch_size=1000,  # Batch size plus grand
        remove_columns=[c for c in train_dataset.column_names if c != "input_ids" and c != "attention_mask" and c != "labels"],
        load_from_cache_file=False  # Désactiver le cache
    )
    if eval_dataset:
        eval_dataset = eval_dataset.map(
            tokenize_and_prepare_labels, 
            batched=True, 
            batch_size=1000,
            remove_columns=[c for c in eval_dataset.column_names if c != "input_ids" and c != "attention_mask" and c != "labels"],
            load_from_cache_file=False
        )

    # ---------------- PEFT (LoRA) config OPTIMISÉ pour vitesse
    # OPTIMISATION: Réduire r pour moins de paramètres (plus rapide)
    peft_config = LoraConfig(
        r=32,  # Réduit de 64 à 32 pour vitesse (moins de paramètres à entraîner)
        lora_alpha=16,  # Garder alpha constant
        lora_dropout=0.05,  # Réduit de 0.1 à 0.05
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        inference_mode=False,
    )
    print(f"📊 LoRA config: r={peft_config.r}, alpha={peft_config.lora_alpha}, dropout={peft_config.lora_dropout}")

    # ---------------- Training args: OPTIMISÉ POUR VITESSE avec Llama-3
    # Calculer les steps optimaux
    total_samples = len(train_dataset)
    train_batch_size = 1  # Réduit à 1 pour éviter OOM
    grad_accum = 32  # Augmenté pour compenser
    effective_batch_size = train_batch_size * grad_accum
    
    # Optimisations pour la vitesse
    save_steps = min(1000, max(100, total_samples // (effective_batch_size * 2)))  # Moins de sauvegardes
    logging_steps = min(100, max(10, total_samples // (effective_batch_size * 10)))  # Moins de logs
    
    # Force no evaluation during training (prevents actor from accumulating logits/tensors)
    eval_steps = None
    
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        # OPTIMISATION: Batch size réduit pour éviter OOM avec Llama-3 8B
        per_device_train_batch_size=1,  # Réduit à 1 pour éviter OOM (était 2)
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=32 if has_cuda else 32,  # Augmenté pour compenser batch_size=1
        # OPTIMISATION: Learning rate légèrement plus élevé pour convergence plus rapide
        learning_rate=3e-4,  # Augmenté de 2e-4 à 3e-4
        optim="paged_adamw_8bit" if has_cuda else "adamw_torch",  # 8-bit au lieu de 32-bit (plus rapide)
        save_steps=save_steps,
        eval_steps=eval_steps,
        logging_steps=logging_steps,
        eval_strategy="no",
        save_total_limit=2,  # Réduire pour économiser I/O
        # OPTIMISATION: Mixed precision plus agressif
        fp16=has_cuda,  # FP16 activé
        bf16=False,
        # OPTIMISATION: Dataloader
        dataloader_pin_memory=has_cuda,
        dataloader_num_workers=2 if has_cuda else 0,  # Workers pour paralléliser le chargement
        dataloader_prefetch_factor=2,  # Précharger les données
        # OPTIMISATION: Autres paramètres
        load_best_model_at_end=False,
        metric_for_best_model=None,
        report_to="none",
        remove_unused_columns=False,
        warmup_steps=min(50, max(10, total_samples // (effective_batch_size * 20))),  # Moins de warmup
        lr_scheduler_type="linear",  # Linear plus simple que cosine
        prediction_loss_only=True,  # Seulement la loss, pas les prédictions complètes
        include_inputs_for_metrics=False,
        eval_accumulation_steps=1,
        # OPTIMISATION: Gradient clipping pour stabilité avec LR plus élevé
        max_grad_norm=1.0,
        # OPTIMISATION: Désactiver certaines vérifications
        ddp_find_unused_parameters=False,
        # OPTIMISATION: Compilation (si disponible)
        # torch_compile=True,  # Décommenter si PyTorch 2.0+
    )

    safe_collator = make_safe_data_collator(tokenizer)

    # ---------------- Trainer: do NOT pass formatting_func when dataset already contains input_ids
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=None,                      # evaluation disabled during training to avoid OOM
        peft_config=peft_config,
        data_collator=safe_collator,
        compute_metrics=None,                   # skip compute_metrics during training
    )

    # ---------------- Train avec gestion améliorée de la mémoire
    print(f"\n🏋️ Début de l'entraînement pour {agent_name}...")
    if has_cuda:
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        # Afficher l'utilisation mémoire avant entraînement
        if torch.cuda.is_available():
            print(f"💾 Mémoire GPU avant entraînement: {torch.cuda.memory_allocated()/1024**3:.2f} GB / {torch.cuda.memory_reserved()/1024**3:.2f} GB")
    
    try:
        train_result = trainer.train()
    except RuntimeError as e:
        error_msg = str(e).lower()
        # Keep exception handling but avoid reapplying PEFT to model (warnings).
        if "out of memory" in error_msg or "oom" in error_msg:
            print(f"\n⚠️ Erreur de mémoire GPU détectée. Nettoyage et réduction des paramètres...")
            torch.cuda.empty_cache()
            import gc
            gc.collect()
            
            # Réduire encore plus les paramètres
            print(f"⚠️ Réduction des paramètres pour éviter OOM...")
            trainer.args.per_device_train_batch_size = 1
            trainer.args.gradient_accumulation_steps = 64  # Augmenter encore plus
            trainer.args.eval_steps = None
            trainer.args.eval_strategy = "no"
            trainer.args.load_best_model_at_end = False
            trainer.args.metric_for_best_model = None
            
            # Réduire la longueur maximale si possible
            print(f"⚠️ Si le problème persiste, réduisez model_max_len à 512")
            
            # Réessayer
            try:
                train_result = trainer.train()
            except RuntimeError as e2:
                print(f"❌ Erreur persistante: {e2}")
                print(f"💡 Solutions:")
                print(f"   1. Réduire model_max_len à 512 dans le code")
                print(f"   2. Réduire la taille du dataset")
                print(f"   3. Utiliser un GPU avec plus de mémoire")
                raise
        else:
            raise

    # ---------------- Métriques finales
    print(f"\n📈 Métriques d'entraînement:")
    print(f"   Loss: {train_result.training_loss:.4f}")
    if hasattr(train_result, "metrics"):
        for key, value in train_result.metrics.items():
            print(f"   {key}: {value:.4f}")

    # ---------------- Évaluation finale si dataset de validation disponible
    # Vérifier si le trainer a un eval_dataset (peut être None après gestion d'erreur OOM)
    has_eval_dataset = hasattr(trainer, 'eval_dataset') and trainer.eval_dataset is not None
    
    if has_eval_dataset:
        print(f"\n🔍 Évaluation finale...")
        try:
            eval_results = trainer.evaluate()
        except ValueError as e:
            if "eval_dataset" in str(e):
                print(f"⚠️ Évaluation finale ignorée : pas de dataset d'évaluation disponible")
                eval_results = {}
            else:
                raise
    else:
        eval_results = {}
        print(f"\n⚠️ Pas d'évaluation finale : dataset d'évaluation non disponible")
        
        # Afficher les métriques de manière organisée
        print(f"\n{'='*70}")
        print(f"📊 RÉSULTATS D'ÉVALUATION - {agent_name.upper()}")
        print(f"{'='*70}")
        
        # Métriques principales (accuracy, F1, etc.)
        print(f"\n🎯 MÉTRIQUES PRINCIPALES:")
        if "eval_token_accuracy" in eval_results:
            print(f"   Accuracy (tokens):     {eval_results['eval_token_accuracy']:.4f}")
        if "eval_exact_match_rate" in eval_results:
            print(f"   Exact Match Rate:      {eval_results['eval_exact_match_rate']:.4f}")
        if "eval_f1_score" in eval_results:
            print(f"   F1 Score:              {eval_results['eval_f1_score']:.4f}")
        if "eval_precision" in eval_results:
            print(f"   Precision:             {eval_results['eval_precision']:.4f}")
        if "eval_recall" in eval_results:
            print(f"   Recall:                {eval_results['eval_recall']:.4f}")
        
        # Métriques de loss
        print(f"\n📉 MÉTRIQUES DE LOSS:")
        if "eval_loss" in eval_results:
            print(f"   Loss:                  {eval_results['eval_loss']:.4f}")
        if "eval_perplexity" in eval_results:
            print(f"   Perplexity:            {eval_results['eval_perplexity']:.4f}")
        
        # Métriques JSON (pour orchestrator)
        if agent_name == "orchestrator":
            print(f"\n📋 MÉTRIQUES JSON:")
            if "eval_json_valid_rate" in eval_results:
                print(f"   JSON Valid Rate:      {eval_results['eval_json_valid_rate']:.4f}")
            if "eval_json_parseable_rate" in eval_results:
                print(f"   JSON Parseable Rate:   {eval_results['eval_json_parseable_rate']:.4f}")
            if "eval_key_match_rate" in eval_results:
                print(f"   Key Match Rate:        {eval_results['eval_key_match_rate']:.4f}")
        
        # Toutes les autres métriques
        other_metrics = {k: v for k, v in eval_results.items() 
                        if k not in ["eval_token_accuracy", "eval_exact_match_rate", "eval_f1_score", 
                                   "eval_precision", "eval_recall", "eval_loss", "eval_perplexity",
                                   "eval_json_valid_rate", "eval_json_parseable_rate", "eval_key_match_rate"]}
        if other_metrics:
            print(f"\n📌 AUTRES MÉTRIQUES:")
            for key, value in other_metrics.items():
                if isinstance(value, (int, float)):
                    print(f"   {key}: {value:.4f}")
        
        print(f"{'='*70}\n")

    # ============================
    # SAUVEGARDE FINALE (OBLIGATOIRE)
    # ============================
    print(f"\n{'='*70}")
    print(f"💾 SAUVEGARDE FINALE DU MODÈLE")
    print(f"{'='*70}")
    print(f"📁 Répertoire de sortie: {output_dir}")
    
    # Sauvegarder le modèle LoRA
    print(f"💾 Sauvegarde du modèle LoRA...")
    trainer.model.save_pretrained(output_dir)
    print(f"✅ Modèle LoRA sauvegardé")
    
    # Sauvegarder le tokenizer
    print(f"💾 Sauvegarde du tokenizer...")
    tokenizer.save_pretrained(output_dir)
    print(f"✅ Tokenizer sauvegardé")
    
    print(f"💾 LoRA sauvegardé dans : {output_dir}")
    print(f"{'='*70}\n")
    
    # Préparer les métriques complètes à sauvegarder
    metrics_to_save = {
        "agent_name": agent_name,
        "base_model": BASE_MODEL_ID,
        "training_config": {
            "num_train_epochs": args.num_train_epochs,
            "per_device_train_batch_size": args.per_device_train_batch_size,
            "per_device_eval_batch_size": args.per_device_eval_batch_size,
            "gradient_accumulation_steps": args.gradient_accumulation_steps,
            "learning_rate": args.learning_rate,
            "optimizer": args.optim,
            "lr_scheduler": args.lr_scheduler_type,
            "warmup_steps": args.warmup_steps,
            "fp16": args.fp16,
            "bf16": args.bf16,
        },
        "peft_config": {
            "r": peft_config.r,
            "lora_alpha": peft_config.lora_alpha,
            "lora_dropout": peft_config.lora_dropout,
            "target_modules": list(peft_config.target_modules) if isinstance(peft_config.target_modules, set) else peft_config.target_modules,
        },
        "dataset_info": {
            "dataset_path": dataset_path,
            "training_samples": len(train_dataset),
            "validation_samples": len(eval_dataset) if eval_dataset else 0,
            "eval_split": eval_split if eval_dataset else 0.0,
        },
        "training_results": {
            "training_loss": float(train_result.training_loss),
            "training_steps": train_result.global_step if hasattr(train_result, 'global_step') else None,
            "training_epochs": train_result.epoch if hasattr(train_result, 'epoch') else None,
        },
    }
    
    # Ajouter les métriques d'évaluation si disponibles
    if eval_results:
        eval_metrics = {}
        for k, v in eval_results.items():
            if isinstance(v, (int, float)):
                eval_metrics[k] = float(v)
        if eval_metrics:
            metrics_to_save["evaluation_results"] = eval_metrics
    
    # Sauvegarder les métriques complètes
    metrics_path = os.path.join(output_dir, "training_metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_to_save, f, indent=2, ensure_ascii=False)
    
    # Créer un fichier README avec un résumé
    readme_path = os.path.join(output_dir, "README.md")
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write(f"# Modèle Fine-Tuné: {agent_name}\n\n")
        f.write(f"## Informations Générales\n\n")
        f.write(f"- **Agent**: {agent_name}\n")
        f.write(f"- **Modèle de base**: {BASE_MODEL_ID}\n")
        f.write(f"- **Date d'entraînement**: {train_result.log_history[-1].get('epoch', 'N/A') if hasattr(train_result, 'log_history') and train_result.log_history else 'N/A'}\n\n")
        f.write(f"## Configuration d'Entraînement\n\n")
        f.write(f"- **Époques**: {args.num_train_epochs}\n")
        f.write(f"- **Batch size (train)**: {args.per_device_train_batch_size}\n")
        f.write(f"- **Batch size (eval)**: {args.per_device_eval_batch_size}\n")
        f.write(f"- **Gradient accumulation**: {args.gradient_accumulation_steps}\n")
        f.write(f"- **Learning rate**: {args.learning_rate}\n")
        f.write(f"- **Optimizer**: {args.optim}\n\n")
        f.write(f"## Configuration LoRA\n\n")
        f.write(f"- **r**: {peft_config.r}\n")
        f.write(f"- **alpha**: {peft_config.lora_alpha}\n")
        f.write(f"- **dropout**: {peft_config.lora_dropout}\n")
        target_modules_list = list(peft_config.target_modules) if isinstance(peft_config.target_modules, set) else peft_config.target_modules
        f.write(f"- **Target modules**: {', '.join(target_modules_list)}\n\n")
        f.write(f"## Données\n\n")
        f.write(f"- **Échantillons d'entraînement**: {len(train_dataset)}\n")
        if eval_dataset:
            f.write(f"- **Échantillons de validation**: {len(eval_dataset)}\n")
        f.write(f"- **Source**: {dataset_path}\n\n")
        f.write(f"## Résultats\n\n")
        f.write(f"### Entraînement\n")
        f.write(f"- **Loss finale**: {train_result.training_loss:.4f}\n")
        if eval_dataset:
            f.write(f"\n### Évaluation\n")
            if "eval_loss" in eval_results:
                f.write(f"- **Loss**: {eval_results['eval_loss']:.4f}\n")
            if "eval_token_accuracy" in eval_results:
                f.write(f"- **Accuracy (tokens)**: {eval_results['eval_token_accuracy']:.4f}\n")
            if "eval_f1_score" in eval_results:
                f.write(f"- **F1 Score**: {eval_results['eval_f1_score']:.4f}\n")
            if "eval_precision" in eval_results:
                f.write(f"- **Precision**: {eval_results['eval_precision']:.4f}\n")
            if "eval_recall" in eval_results:
                f.write(f"- **Recall**: {eval_results['eval_recall']:.4f}\n")
        f.write(f"\n## Fichiers Sauvegardés\n\n")
        f.write(f"- `adapter_model.bin` ou `adapter_model.safetensors`: Poids LoRA\n")
        f.write(f"- `adapter_config.json`: Configuration LoRA\n")
        f.write(f"- `tokenizer.json`, `tokenizer_config.json`: Tokenizer\n")
        f.write(f"- `training_metrics.json`: Métriques complètes (JSON)\n")
        f.write(f"- `README.md`: Ce fichier\n")
        if os.path.exists(os.path.join(output_dir, "training_args.bin")):
            f.write(f"- `training_args.bin`: Arguments d'entraînement\n")
        f.write(f"\n## Utilisation\n\n")
        f.write(f"```python\n")
        f.write(f"from peft import PeftModel\n")
        f.write(f"from transformers import AutoModelForCausalLM, AutoTokenizer\n\n")
        f.write(f"base_model = AutoModelForCausalLM.from_pretrained('{BASE_MODEL_ID}')\n")
        f.write(f"model = PeftModel.from_pretrained(base_model, '{output_dir}')\n")
        f.write(f"tokenizer = AutoTokenizer.from_pretrained('{output_dir}')\n")
        f.write(f"```\n")
    
    # Afficher le résumé final
    print(f"\n{'='*70}")
    print(f"✅ SFT TERMINÉ pour {agent_name}")
    print(f"{'='*70}")
    print(f"\n📁 Fichiers sauvegardés dans: {output_dir}")
    print(f"\n📄 Contenu du répertoire:")
    saved_files = os.listdir(output_dir)
    for file in sorted(saved_files):
        file_path = os.path.join(output_dir, file)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path)
            size_mb = size / (1024 * 1024)
            print(f"   - {file} ({size_mb:.2f} MB)")
        else:
            print(f"   - {file}/ (dossier)")
    
    print(f"\n📊 Métriques principales:")
    print(f"   Training Loss: {train_result.training_loss:.4f}")
    if eval_results:
        if "eval_token_accuracy" in eval_results:
            print(f"   Accuracy: {eval_results['eval_token_accuracy']:.4f}")
        if "eval_f1_score" in eval_results:
            print(f"   F1 Score: {eval_results['eval_f1_score']:.4f}")
        if "eval_loss" in eval_results:
            print(f"   Eval Loss: {eval_results['eval_loss']:.4f}")
    
    print(f"\n📝 Documentation: {readme_path}")
    print(f"📈 Métriques complètes: {metrics_path}")
    print(f"{'='*70}\n")
    
    return {
        "output_dir": output_dir,
        "metrics": metrics_to_save,
        "model": trainer.model,
        "tokenizer": tokenizer,
    }



In [ ]:
# ---------------- Run for all agents (paths RELATIFS au CWD) ----------------
# Vous pouvez exécuter un agent spécifique ou tous les agents

# Option 1: Entraîner un agent spécifique
# agent_name = "orchestrator"  # ou "researcher", "code_writer", "critic"
# ds = f"data/processed_sft/{agent_name}_sft.jsonl"
# out = f"checkpoints/{agent_name}_lora"
# os.makedirs(out, exist_ok=True)
# run_sft_training(ds, out, agent_name, eval_split=0.1)

# ---------------- Détection automatique de l'environnement (Kaggle vs Local)
def detect_environment():
    """Détecte si on est sur Kaggle ou en local"""
    is_kaggle = os.path.exists("/kaggle/input")
    return is_kaggle

IS_KAGGLE = detect_environment()

if IS_KAGGLE:
    print("🌐 Environnement détecté: KAGGLE")
    print("📁 Utilisation des chemins Kaggle")
    
    # Chercher les datasets dans différents emplacements possibles
    possible_paths = [
        "/kaggle/input/dataset/processed_sft",  # Structure avec sous-dossier
        "/kaggle/input/dataset",  # Structure directe
        "/kaggle/input/processed_sft",  # Autre structure possible
    ]
    
    DATA_BASE = None
    for path in possible_paths:
        if os.path.exists(path):
            # Vérifier qu'il y a au moins un fichier .jsonl
            files = [f for f in os.listdir(path) if f.endswith('.jsonl')]
            if files:
                DATA_BASE = path
                print(f"   ✅ Dataset trouvé dans: {path}")
                print(f"   📄 Fichiers trouvés: {len(files)}")
                break
    
    if DATA_BASE is None:
        print("   ⚠️  Aucun dataset trouvé dans les emplacements standards")
        print("   💡 Vérifiez la structure de votre dataset Kaggle")
        print("   💡 Les datasets doivent être dans /kaggle/input/")
        print("   💡 Liste des dossiers dans /kaggle/input/:")
        if os.path.exists("/kaggle/input"):
            for item in os.listdir("/kaggle/input"):
                item_path = os.path.join("/kaggle/input", item)
                if os.path.isdir(item_path):
                    print(f"      - {item}/")
                    # Lister quelques fichiers
                    try:
                        files = os.listdir(item_path)[:5]
                        for f in files:
                            print(f"        └─ {f}")
                    except:
                        pass
        # Utiliser le premier chemin par défaut
        DATA_BASE = "/kaggle/input/dataset/processed_sft"
        print(f"   ⚠️  Utilisation du chemin par défaut: {DATA_BASE}")
    
    CHECKPOINTS_BASE = "/kaggle/working/checkpoints"
else:
    print("💻 Environnement détecté: LOCAL")
    print("📁 Utilisation des chemins locaux")
    DATA_BASE = "data/processed_sft"
    CHECKPOINTS_BASE = "checkpoints"

print(f"\n   📂 Data base: {DATA_BASE}")
print(f"   💾 Checkpoints base: {CHECKPOINTS_BASE}\n")

# Option 2: Entraîner tous les agents
agents = ["orchestrator", "researcher", "code_writer", "critic"]
results_summary = {}

for agent in agents:
    ds = os.path.join(DATA_BASE, f"{agent}_sft.jsonl")
    out = os.path.join(CHECKPOINTS_BASE, f"{agent}_lora")
    os.makedirs(out, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"📋 Agent: {agent}")
    print(f"   Dataset: {ds}")
    print(f"   Output: {out}")
    print(f"{'='*70}\n")
    
    try:
        run_sft_training(ds, out, agent, eval_split=0.1)
        results_summary[agent] = "✅ Success"
    except Exception as e:
        print(f"\n❌ Erreur durant l'entraînement de {agent} ---")
        print(f"   {type(e).__name__}: {e}")
        results_summary[agent] = f"❌ Failed: {str(e)[:100]}"
        import traceback
        traceback.print_exc()
        continue

# ---------------- Résumé final
print("\n" + "="*70)
print("📊 RÉSUMÉ DE L'ENTRAÎNEMENT")
print("="*70)
for agent, status in results_summary.items():
    print(f"   {agent:15s}: {status}")
print("="*70)
print("✅ Fin du script d'entraînement")
print("="*70)